In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('ispu_dki_all.csv')

print(f"Jumlah baris dan kolom awal: {df.shape}")
print("Missing value awal:\n", df.isnull().sum())

Jumlah baris dan kolom awal: (5538, 11)
Missing value awal:
 tanggal        0
stasiun        1
pm25        4022
pm10         315
so2          130
co            88
o3           104
no2          106
max            1
critical       4
categori       0
dtype: int64


In [3]:
df = df[df['categori'] != 'TIDAK ADA DATA']

if 'pm25' in df.columns:
    df.drop('pm25', axis=1, inplace=True)

if 'max' in df.columns:
    df.drop('max', axis=1, inplace=True)
num_cols = ['pm10', 'so2', 'co', 'o3', 'no2']

for col in num_cols:
    df[col] = df.groupby('stasiun')[col].transform(lambda x: x.fillna(x.median()))
    df[col].fillna(df[col].median(), inplace=True)
df['critical'] = df['critical'].fillna('Tidak Terdeteksi')

print(f"\nJumlah baris setelah dibersihkan: {df.shape}")
print("Missing value setelah cleaning:\n", df.isnull().sum())


Jumlah baris setelah dibersihkan: (5534, 9)
Missing value setelah cleaning:
 tanggal     0
stasiun     1
pm10        0
so2         0
co          0
o3          0
no2         0
critical    0
categori    0
dtype: int64


In [4]:
df['tanggal'] = pd.to_datetime(df['tanggal'])

df['tahun'] = df['tanggal'].dt.year
df['bulan'] = df['tanggal'].dt.month
df['hari'] = df['tanggal'].dt.day
df['hari_dalam_minggu'] = df['tanggal'].dt.dayofweek

df.drop('tanggal', axis=1, inplace=True)

In [5]:
le = LabelEncoder()
df['categori_encoded'] = le.fit_transform(df['categori'])
print(f"Mapping Label Target: {dict(zip(le.classes_, le.transform(le.classes_)))}")

df = pd.get_dummies(df, columns=['critical', 'stasiun'], drop_first=True, dtype=int)
X = df.drop(['categori', 'categori_encoded'], axis=1)
y = df['categori_encoded']

Mapping Label Target: {'BAIK': np.int64(0), 'BERBAHAYA': np.int64(1), 'SANGAT TIDAK SEHAT': np.int64(2), 'SEDANG': np.int64(3), 'TIDAK SEHAT': np.int64(4)}


In [6]:
def cap_outliers_iqr(df_feature):
    for col in df_feature.columns:
        if df_feature[col].dtype in ['int64', 'float64', 'int32']:
            Q1 = df_feature[col].quantile(0.25)
            Q3 = df_feature[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            df_feature[col] = np.where(df_feature[col] < lower_bound, lower_bound, df_feature[col])
            df_feature[col] = np.where(df_feature[col] > upper_bound, upper_bound, df_feature[col])
    return df_feature

X = cap_outliers_iqr(X)

In [7]:
scaler = RobustScaler()

original_num_cols = ['pm10', 'so2', 'co', 'o3', 'no2', 'tahun', 'bulan', 'hari', 'hari_dalam_minggu']

X[original_num_cols] = scaler.fit_transform(X[original_num_cols])

In [10]:
print("PREPROCESSING SELESAI!")
print(f"Dimensi Fitur (X): {X.shape}")
print(f"Dimensi Target (y): {y.shape}")
print("\nSampel 5 baris pertama dari Fitur (X):")
display(X.head())

PREPROCESSING SELESAI!
Dimensi Fitur (X): (5534, 18)
Dimensi Target (y): (5534,)

Sampel 5 baris pertama dari Fitur (X):


,pm10,so2,co,o3,no2,tahun,bulan,hari,hari_dalam_minggu,critical_NO2,critical_O3,critical_PM10,critical_PM25,critical_SO2,stasiun_DKI2 (Kelapa Gading),stasiun_DKI3 (Jagakarsa),stasiun_DKI4 (Lubang Buaya),stasiun_DKI5 (Kebon Jeruk)
0,0.15,-0.7,2.038462,-0.621212,0.2,-0.875,-0.833333,-1.000000,0.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.25,-0.8,-0.153846,-0.530303,-0.3,-0.875,-0.833333,-0.933333,0.50,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1.50,-0.8,0.076923,-0.727273,-0.3,-0.875,-0.833333,-0.866667,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1.75,-0.8,-0.153846,-0.803030,-0.6,-0.875,-0.833333,-0.800000,-0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1.60,-0.8,-0.076923,-0.803030,-0.4,-0.875,-0.833333,-0.733333,-0.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
import pickle

data_bundle = {
    'X': X,
    'y': y,
    'le': le
}

with open('ispu_preprocessed.pkl', 'wb') as file:
    pickle.dump(data_bundle, file)

from google.colab import files
files.download('ispu_preprocessed.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>